In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['OPEN_AI_KEY'] = os.getenv("OPENAI_API_KEY")

In [3]:
from langchain_openai import OpenAIEmbeddings

## https://docs.langchain.com/oss/python/integrations/text_embedding

In [4]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",api_key=os.environ['OPEN_AI_KEY'])

In [4]:
embeddings.embed_query("Hello world")

[-0.00877454411238432,
 -0.010246319696307182,
 0.006171961780637503,
 0.031714390963315964,
 0.008256617933511734,
 -0.0063618686981499195,
 -0.004193049855530262,
 0.07665317505598068,
 0.027381068095564842,
 0.029021169990301132,
 0.0019983346574008465,
 -0.011627458035945892,
 -0.020147355273365974,
 -0.01923235133290291,
 -0.005045470781624317,
 0.036979980766773224,
 -0.012654679827392101,
 -0.002714799949899316,
 -0.007311400957405567,
 -0.018161969259381294,
 0.02225359156727791,
 0.003765759989619255,
 -0.017574984580278397,
 0.05507289245724678,
 0.002958657220005989,
 0.024066336452960968,
 -0.014683227054774761,
 0.005943210795521736,
 -0.036047711968421936,
 -0.027933523058891296,
 0.006159013602882624,
 0.016521867364645004,
 0.010936888866126537,
 0.014398367144167423,
 0.023427559062838554,
 0.005852573551237583,
 0.02520577423274517,
 0.01845546066761017,
 0.005144740454852581,
 -0.0028615461196750402,
 0.03546072542667389,
 0.022374441847205162,
 -0.01720380410552025,

In [23]:
len(embeddings.embed_query("Hello world"))

3072

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

documents = ["Hello world", "Hello", "World", "Hi there"]


my_query = "Hello world"


query_embedding = embeddings.embed_query(my_query)
document_embedding = embeddings.embed_documents(documents)

In [30]:
scores = cosine_similarity([query_embedding], document_embedding)
scores

array([[1.        , 0.64219981, 0.43618991, 0.49166417]])

## FAISS

In [8]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import  InMemoryDocstore

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x103e61c70>>
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


In [9]:

index = faiss.IndexFlatIP(3072)

In [7]:
vectorstore = FAISS(embedding_function=embeddings, index=index, docstore=InMemoryDocstore({}), index_to_docstore_id={})

In [8]:
vectorstore

In [ ]:
vectorstore.add_texts(["AI is powerful", "AI is the future", "computer science is facinating"], [{"source": "doc1"}, {"source": "doc2"}, {"source": "doc3"}])

['f6eb0985-b3d9-4461-acb2-e1a8bfab31a8',
 '91e279bc-7e47-4d37-b457-aa3c12fb3c04',
 'b0d5c0d2-6e00-405f-a627-6b3880803e5f']

In [10]:
vectorstore.index_to_docstore_id

{0: 'f6eb0985-b3d9-4461-acb2-e1a8bfab31a8',
 1: '91e279bc-7e47-4d37-b457-aa3c12fb3c04',
 2: 'b0d5c0d2-6e00-405f-a627-6b3880803e5f'}

In [11]:
vectorstore.similarity_search("What is the future of AI?", k=2)

[Document(id='91e279bc-7e47-4d37-b457-aa3c12fb3c04', metadata={'source': 'doc2'}, page_content='AI is the future'),
 Document(id='f6eb0985-b3d9-4461-acb2-e1a8bfab31a8', metadata={'source': 'doc1'}, page_content='AI is powerful')]

## Filtering with metadata

In [13]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [14]:
vectorstore.add_documents(documents)

['98589d04-2b01-4123-9295-995ba28019f9',
 '938123e4-9479-48ec-8cbc-2a262cf6041b',
 'b82023ef-66b5-47b7-86d0-f87a41cbe48d',
 '83d048d9-ddb8-4eb3-9c29-fe8e6d23f580',
 '7c5eb834-fd94-43b2-86b1-3ba23e3dfc84',
 'f52338d3-e2dc-4f35-8927-af2e36418208',
 '07586852-b576-4ee6-94fd-a48189ac57ec',
 'dc2fa371-e4a7-43c4-ae1e-1fb4ff6a9abb',
 'd4cad4b0-4d63-4510-bda8-0cd1fe306c0d',
 'ffa0c35d-3700-4ea7-a23d-3a18717992b3']

In [17]:
vectorstore.similarity_search("What is the future of AI?", k=2, filter={"source": "tweet"})

[Document(id='dc2fa371-e4a7-43c4-ae1e-1fb4ff6a9abb', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='b82023ef-66b5-47b7-86d0-f87a41cbe48d', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!')]

In [18]:
vectorstore.similarity_search("Tell about Langchain", k=2, filter={"source": "tweet"})

[Document(id='b82023ef-66b5-47b7-86d0-f87a41cbe48d', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='dc2fa371-e4a7-43c4-ae1e-1fb4ff6a9abb', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [21]:
vectorstore.save_local("/Users/cardinality/Documents/BIA GIAI/bia-gndy-jan-genai/RAG/faiss_index")

## PDF Chunking

In [6]:
file_path = "/Users/cardinality/Documents/BIA GIAI/bia-gndy-jan-genai/RAG/llama2.pdf"

In [7]:
from langchain_community.document_loaders import PyPDFLoader

In [10]:
pages =[]
for page in PyPDFLoader(file_path).load():
    #print(page.page_content)
    pages.append(page)

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

split_docs = splitter.split_documents(pages)
split_docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/cardinality/Documents/BIA GIAI/bia-gndy-jan-genai/RAG/llama2.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}, page_content='Llama 2 : Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗Louis Martin†Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Ar

In [12]:
vectorstore1 = FAISS(embedding_function=embeddings, index=index, docstore=InMemoryDocstore({}), index_to_docstore_id={})

In [13]:
vectorstore1.add_documents(documents=split_docs)

['594d9570-b1c9-4391-bddd-f4b4a29624ce',
 '298cc858-7326-4868-91b4-9db7404335da',
 '323df1c0-e68f-4347-b3f1-7f7ed71107d0',
 'f400c952-39ae-4356-9281-78ff961dbecf',
 '887f66ec-bfcb-4fe9-a764-ba97fcab8688',
 '72ec67a9-3416-4dfe-aa23-517ca685405d',
 '48d3c174-24e2-4ff7-ab7e-4fdbef8890c2',
 'aedd2175-6edc-4699-9cbe-6804ce732dd6',
 '9b567e1c-afd0-442b-a20d-6078be35506b',
 '1de18f4d-7fd7-4aad-8c38-d463b9e79ad2',
 'de0754ef-efe5-438c-b3c6-54d1074f20f1',
 '00dbbf95-79bd-4b65-8135-2ad986c3af0a',
 'f9fb6ef3-7a8b-4e72-9672-92cc4abb28e2',
 '3cddadfd-81d6-44d9-a039-eaf7eb89a4c0',
 '754473b4-2dd8-47f1-ab24-554222b53bd0',
 'e5d89a8c-a1d6-4e17-9825-647c878781f9',
 'abf462b7-c7bb-40cd-bd38-b7786776ec54',
 'c4b8ebc1-1bb8-497c-af32-ec49a2a07d53',
 'add97410-7f78-4537-b9d3-3fbbec267e94',
 '3447afac-9328-4039-bf7e-6fbe9988bbb5',
 '8e8cb65f-b3eb-4a02-8b6d-cbb027b097b4',
 '684f67cd-129d-487d-8c7b-805a427e4587',
 '05e72409-a45a-4d9b-8cdf-27bfb3ef89bf',
 '3aad3952-a78b-4bf0-a132-9b1dc7ac2f30',
 '2d97e8d0-7f20-

In [14]:
retriver = vectorstore1.as_retriever(search_kwargs={"k": 2})

In [15]:
retrive_result = retriver.invoke("what is llama2?")
for result in retrive_result:
    print(result.page_content)

principlesofhelpfulnessandsafety. Tocontributemoresignificantlytosocietyandfosterthepaceofresearch,
wehaveresponsiblyopenedaccessto Llama 2 andLlama 2-Chat . Aspartofourongoingcommitmentto
transparency and safety, we plan to make further improvements to Llama 2-Chat in future work.
36
Llama 2 is a new technology that carries risks with use. Testing conducted to date has been in
English, and has notcovered, nor could it coverall scenarios. For these reasons, aswith all LLMs,
Llama 2’s potential outputs cannot be predicted in advance, and the model may in some instances
produceinaccurateorobjectionableresponsestouserprompts. Therefore,beforedeployingany
applications of Llama 2, developers should perform safety testing and tuning tailored to their


In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [19]:
llm = ChatOpenAI(model="gpt-3.5-turbo", api_key=os.environ['OPEN_AI_KEY'])

In [20]:
prompt = ChatPromptTemplate.from_template(''' Your are a helpful assistant that answers questions about the document provided.
Use only the information from the document to answer the question. If you don't know the answer, say you don't know.

Context: {context}

Question: {question}

Answer: ''')

In [21]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [22]:
rag_chain = ( { "context" :retriver | format_docs , "question" : RunnablePassthrough() } | prompt | llm | StrOutputParser() )

In [23]:
query ="what is llama2?"
response = rag_chain.invoke(query)
print(response)

Llama 2 is a new technology that carries risks with its use. It is an LLM (Large Language Model) that may produce inaccurate or objectionable responses to user prompts. Developers should perform safety testing and tuning before deploying any applications of Llama 2.
